# 05 adapter 保存、加载和固定 prompt 对比：怎么判断真的变好

训练完不等于成功。这个项目最重要的工程习惯是：训练、保存、加载、固定 prompt 对比，缺一不可。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap, re

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)
print("学习原则: 本 notebook 默认只读项目文件；所有真实推理/训练开关默认 False。")

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=None):
    lines = read_text(rel).splitlines()
    if end is None:
        end = len(lines)
    print(f"--- {rel}:{start}-{end} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = []
    for idx, line in enumerate(lines, start=1):
        if keyword in line:
            hits.append(idx)
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        start = max(1, hit - context)
        end = min(len(lines), hit + context)
        show_file(rel, start, end)
        print()

def read_jsonl(rel, n=3):
    rows = []
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

## 1. SFT adapter 怎么保存？

In [ ]:
find_lines("scripts/train_sft_lora.py", "trainer.save_model", context=8)
find_lines("scripts/train_sft_lora.py", "tokenizer.save_pretrained", context=6)

LoRA 输出目录里最重要的是 `adapter_model.safetensors`、`adapter_config.json` 和 tokenizer 文件。

In [ ]:
adapter = PROJECT_ROOT / "outputs/sft_lora_qwen05b_custom_v3_from_v1_patch"
print("adapter exists:", adapter.exists())
if adapter.exists():
    for p in adapter.iterdir():
        if p.name in ["adapter_model.safetensors", "adapter_config.json", "tokenizer_config.json", "README.md"]:
            print(p.name, p.stat().st_size)

## 2. 推理时 adapter 怎么加载？

In [ ]:
find_lines("scripts/infer.py", "adapter_path", context=10)
find_lines("scripts/infer.py", "PeftModel.from_pretrained", context=8)

推理流程是：加载 base Qwen，如果传入 adapter_path，就用 PeftModel.from_pretrained 挂上 LoRA adapter，然后 generate 回答。所以 adapter 不是完整模型，离不开 base Qwen。

## 3. 为什么要固定 prompt 对比？

In [ ]:
find_lines("scripts/compare_three_outputs.py", "run_infer", context=18)

这个脚本每个 prompt 跑三次：base_answer、public_sft_answer、custom_sft_answer。它故意用 subprocess 分开跑，是为了低显存：一次只加载一个模型/adapter。

## 4. 固定 prompt 是项目的考试卷

In [ ]:
rows = read_jsonl("data/samples/custom_technical_prompts.jsonl", n=8)
for i, row in enumerate(rows, start=1):
    print(i, row["prompt"])

这 8 个 prompt 覆盖 LoRA、SFT、DPO、数据管线、显存风险、loss 与行为关系、面试讲述。项目后期最难的是第 7 题。

## 5. 自动打分为什么只是辅助？

In [ ]:
find_lines("scripts/score_fixed_prompt_outputs.py", "RUBRICS", context=12)
find_lines("scripts/score_fixed_prompt_outputs.py", "criterion_hit", context=8)

自动打分用关键词规则检查必要概念和禁用错误。它可复现，但不如人类灵活。所以它适合作为行为门禁辅助，而不是唯一裁判。

## 6. 你要能讲出的底层句子

> adapter 保存后必须重新加载验证；模型是否变好不能只看训练 loss，要用同一批固定 prompt 比较 base、public-SFT、custom-SFT 和 DPO 输出，检查目标概念是否修复以及旧能力是否回归。